# ISO TC/211 Harmonised Model Definitions

Connect to the EA app and model repository

In [1]:
from Parameters import *
from EAConnect import *
from HM_Controls import *
import sys
import pandas as pd

# Open EA Repository and find Model
eaApp = openEAapp()
eaRepo = openEArepo(eaApp,repo_path)
try:
    omMod = eaRepo.Models.GetByName(modelName)
    printTS('Model "' + modelName + '" found with PackageGUID ' + omMod.PackageGUID )
except Exception as e:
    printTS('Model  "' + modelName + '" not found!')
    closeEA(eaRepo)
    sys.exit()
printTS('Number of main packages: ' + str(omMod.Packages.Count))
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

2026-06-23 09:25:11   Hi EA - are you there? 
2026-06-23 09:25:11   I am here
2026-06-23 09:25:11   Hi EA - Please open this repository: C:\Data\GitHub\ISO TC211\HMMG\EditorialVersion\ISOTC211_HM EditorialVersion.qea
2026-06-23 09:25:13   OK! Repository C:\Data\GitHub\ISO TC211\HMMG\EditorialVersion\ISOTC211_HM EditorialVersion.qea is ready!
2026-06-23 09:25:13   Model "Conceptual Models" found with PackageGUID {7B6B28E9-C583-4363-9E9C-F37A37AE06C9}
2026-06-23 09:25:13   Number of main packages: 5


## Select a package in EA before continuing

In [2]:
thePackage = eaRepo.GetTreeSelectedPackage()
printTS('Selected package name: ' + thePackage.Name)

2026-06-23 09:25:21   Selected package name: ISO/CD TR 19121 Edition 2


**Definitions:**

List all definitions for classes, attributes and navigable associations

In [3]:
#defDf = pd.DataFrame(columns=['Type','PackageName','ElementName','PropertyName','Supplier'])
defDf = listAllDefinitions(eaRepo,thePackage)

printTS('Count:' + str(len(defDf)))

2026-06-23 09:25:25   Count:61


Restore package to baseline and reselect package before continuing

In [ ]:
# Loop through defDf
for index, row in defDf.iterrows():
    # printTS('Type: ' + row["Type"])
    if row['Type'].upper() in ["CLASS","INTERFACE", "DATATYPE","ENUMERATION"]:
        try:
            eaEl = eaRepo.GetElementByGuid(str(row['GUID']))
            eaEl.Notes = row['Definition']
            eaEl.Update()
            # printTS(eaEl.Name + ': ' + row['Definition'])
        except Exception as e:
            printTS('ERROR|' + str(e)) 
    if row['Type'].upper() == "ATTRIBUTE": 
        try:
            eaAttr = eaRepo.GetAttributeByGuid(str(row['GUID']))
            eaAttr.Notes = row['Definition']
            eaAttr.Update()
        except Exception as e:
            printTS('ERROR|' + str(e)) 
    if row['Type'].upper() == "ROLE":
        try:
            eaCon = eaRepo.GetConnectorByGuid(str(row['GUID']))
            # eaAttr.Notes = row['Definition']
            # eaAttr.Update()
        except Exception as e:
            printTS('ERROR|' + str(e))

# A bit more here...

In [ ]:
#defDf = pd.DataFrame(columns=['Type','PackageName','ElementName','PropertyName','Supplier'])
defDf = listMissingDefinitions(eaRepo,thePackage)

errCount = len(defDf)
printTS('')
printTS('Number of errors: ' + str(errCount))
print('')


noDef = defDf[defDf['PropertyName'].isna() & defDf['Supplier'].isna()]
if len(noDef) > 0:
    print('')
    print('Elements without definitions (' + str(len(noDef)) + '):')
    for index, row in noDef.iterrows():
        combined_string = f"{row['PackageName']}.{row['ElementName']}"
        print(combined_string)

noDef = defDf[defDf['Type']=='Attribute']
if len(noDef) > 0:
    print('')
    print('Attributes without definitions (' + str(len(noDef)) + '):')
    for index, row in noDef.iterrows():
        combined_string = f"{row['PackageName']}.{row['ElementName']}.{row['PropertyName']}"
        print(combined_string)

noDef = defDf[defDf['Type']=='Code value']
if len(noDef) > 0:
    print('')
    print('Code values without definitions (' + str(len(noDef)) + '):')
    for index, row in noDef.iterrows():
        combined_string = f"{row['PackageName']}.{row['ElementName']}.{row['PropertyName']}"
        print(combined_string)

noDef = defDf[defDf['Type']=='Role name']
if len(noDef) > 0:
    print('')
    print('Navigable association ends without role name (' + str(len(noDef)) + '):')
    for index, row in noDef.iterrows():
        combined_string = f"{row['PackageName']}.{row['ElementName']} towards {row['Supplier']}"
        print(combined_string)

noDef = defDf[defDf['Type']=='Role']
if len(noDef) > 0:
    print('')
    print('Navigable association ends without definition (' + str(len(noDef)) + '):')
    for index, row in noDef.iterrows():
        if row['PropertyName'] != None:
            combined_string = f"{row['PackageName']}.{row['ElementName']}.{row['PropertyName']} towards {row['Supplier']}"
        else:
            combined_string = f"{row['PackageName']}.{row['ElementName']} towards {row['Supplier']}"
        print(combined_string)


2025-03-25 11:05:16   ----------------------------
2025-03-25 11:05:16   Package: PK
2025-03-25 11:05:16   ----------------------------
2025-03-25 11:05:16   Package: B2GM
2025-03-25 11:05:16   ----------------------------
2025-03-25 11:05:16   Package: B2GM EM
2025-03-25 11:05:16   ERROR|Missing role name for Navigable association in package B2GM EM: EM_rule towards EM_destination
2025-03-25 11:05:16   ERROR|Missing role definition for Navigable association in package B2GM EM: EM_rule towards EM_destination
2025-03-25 11:05:16   ERROR|Missing role name for Navigable association in package B2GM EM: EM_rule towards EM_source
2025-03-25 11:05:16   ERROR|Missing role definition for Navigable association in package B2GM EM: EM_rule towards EM_source
2025-03-25 11:05:16   ERROR|Missing role name for Navigable association in package B2GM EM: EM_ruleset towards EM_rule
2025-03-25 11:05:16   ERROR|Missing role definition for Navigable association in package B2GM EM: EM_ruleset towards EM_rule


Export report

In [ ]:
print('')
file_path = mainFolder + '\\Definitions.xlsx'
# Export to Excel 
writer = pd.ExcelWriter(file_path)

noDef = defDf[defDf['Type']!='Role name']
if len(noDef) > 0:
    noDef.to_excel(writer,'Definitions') 
    print(f"Exported definitions report to file: {file_path}")   

noDef = defDf[defDf['Type']=='Role name']
if len(noDef) > 0:
    noDef.to_excel(writer,'Role names') 
    print(f"Exported role names report to file: {file_path}") 
   

writer.close()



Exported definitions report to file: C:\Data\GitHub\ISO TC211\HMMG\EditorialVersion\Definitions.xlsx
Exported role names report to file: C:\Data\GitHub\ISO TC211\HMMG\EditorialVersion\Definitions.xlsx


C:\Users\JETKNU\AppData\Local\Temp\ipykernel_18568\1554093949.py:8: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  noDef.to_excel(writer,'Definitions')
C:\Users\JETKNU\AppData\Local\Temp\ipykernel_18568\1554093949.py:13: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  noDef.to_excel(writer,'Role names')
